In [ ]:
import pandas as pd
import random
from sklearn.model_selection import train_test_split
import pathlib
from PIL import Image
import numpy as np
from tqdm import tqdm
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Projektpfade relativ zum Notebooks-Ordner aufsetzen
repo_root = pathlib.Path().resolve().parent  # notebooks -> repo root
data_dir = repo_root / "data"
models_dir = repo_root / "models"
features_dir = data_dir / "features"
splits_dir = data_dir / "splits"

models_dir.mkdir(parents=True, exist_ok=True)
features_dir.mkdir(parents=True, exist_ok=True)
splits_dir.mkdir(parents=True, exist_ok=True)

#### Vorgefertigte Feature-Vektoren laden

Train-Features sind bereits undersampled (aus 02_splits). Val/Test bleiben vollständig.

In [ ]:
# Vorgefertigte Feature-Vektoren laden
feat_train = np.load(features_dir / "dt_features.npz", allow_pickle=True)
X_train, y_train = feat_train["X_train"], feat_train["y_train"]
X_val, y_val = feat_train["X_val"], feat_train["y_val"]
X_test, y_test = feat_train["X_test"], feat_train["y_test"]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# Klassenverteilung
for name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    unique, counts = np.unique(labels, return_counts=True)
    dist = dict(zip(unique, counts))
    print(f"{name} Klassenverteilung: {dist}")

## 1. Decision Tree (Ungeprunter Baum – Baseline)

Wir trainieren einen Decision Tree mit max_depth=20 direkt auf den Pixel-Features. Dies ist die Baseline-Variante ohne jegliche Pruning-Strategie. Decision Trees benötigen keine Dimensionalitätsreduktion wie PCA, da sie automatisch relevante Features selektieren können.

Trainiert den Decision Tree mit festen Hyperparametern auf den Pixel-Features.

In [ ]:
# Decision Tree (Ungeprunter Baum – Baseline)
dt_classifier = DecisionTreeClassifier(
    random_state=42,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5
)
print("Trainiere Decision Tree...")
dt_classifier.fit(X_train, y_train)
print("Decision Tree trainiert!")

# Vorhersagen
y_val_pred = dt_classifier.predict(X_val)
y_test_pred = dt_classifier.predict(X_test)

val_accuracy = accuracy_score(y_val, y_val_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"\nValidierungs-Genauigkeit: {val_accuracy:.4f}")
print(f"Test-Genauigkeit: {test_accuracy:.4f}")

# Genauigkeit
val_accuracy = accuracy_score(y_val, y_val_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"\nValidierungs-Genauigkeit: {val_accuracy:.4f}")
print(f"Test-Genauigkeit:         {test_accuracy:.4f}")

Evaluierung: Accuracy, Classification Report und Confusion Matrix auf dem Test-Set.

In [ ]:
# Evaluierung: Details
print("\n" + "="*60)
print("CLASSIFICATION REPORT - Test Set")
print("="*60)
print(classification_report(y_test, y_test_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

# Visualisierung der Confusion Matrix
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[str(i) for i in np.unique(y_test)],
            yticklabels=[str(i) for i in np.unique(y_test)])
plt.title("Confusion Matrix auf Testset", fontsize=10)
plt.xlabel("Predicted", fontsize=10)
plt.ylabel("True", fontsize=10)
plt.tight_layout()
plt.show()


print(f"\nTest Accuracy: {test_accuracy:.4f}")

### Feature Importance

Feature-Importance als 32×32-Heatmap: zeigt genutzte Bildbereiche des Baums.

Die wichtigsten Features für die Klassifikation werden anhand der Reduktion des Gini-Index bestimmt:

In [ ]:
# Feature-Importance darstellen
feat_importance = dt_classifier.feature_importances_

# Reshape zu 32x32 Bild
feat_importance_img = feat_importance.reshape(32, 32)

# Heatmap
plt.figure(figsize=(4, 3))
sns.heatmap(feat_importance_img, cmap='hot', cbar_kws={'label': 'Importance'})
plt.title(f'Feature Importance Heatmap\n(Welche Bildbereiche nutzt der Decision Tree?)')
plt.xlabel('Pixel X')
plt.ylabel('Pixel Y')
plt.tight_layout()
plt.show()

# Statistik
print(f"Durchschnittliche Feature Importance: {feat_importance.mean():.6f}")
print(f"Max Feature Importance: {feat_importance.max():.6f}")
print(f"Features mit Importance > 0: {np.sum(feat_importance > 0)} von {len(feat_importance)}")



## 2. Pre-Pruning: Optimale Baumtiefe

Pre-Pruning durch Hyperparameter-Tuning: Sweep über max_depth (1–20) zur Findung der optimalen Tiefe basierend auf Validierungs-Accuracy.

Dies ist ein proaktiver Ansatz: Der Baum wird von Anfang an mit begrenzter Tiefe trainiert, um Overfitting zu vermeiden:

In [ ]:
# Hyperparameter-Tuning: max_depth variieren (begrenzt auf 20 für Speed)
max_depths = range(1, 21)
train_accuracies = []
val_accuracies = []

for depth in max_depths:
    dt_temp = DecisionTreeClassifier(
        random_state=42,
        max_depth=depth,
        min_samples_split=10,
        min_samples_leaf=5
    )
    dt_temp.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, dt_temp.predict(X_train))
    val_acc = accuracy_score(y_val, dt_temp.predict(X_val))
    
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

# Visualisierung
plt.figure(figsize=(12, 6))
plt.plot(max_depths, train_accuracies, marker='o', label='Training', linewidth=2)
plt.plot(max_depths, val_accuracies, marker='s', label='Validation', linewidth=2)
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree: Accuracy vs Max Depth (bis 20)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Beste Tiefe basierend auf Validierungsdaten
best_depth = max_depths[np.argmax(val_accuracies)]
best_val_acc = max(val_accuracies)
print(f"\nBeste max_depth: {best_depth} mit Validierungs-Accuracy: {best_val_acc:.4f}")

# Test-Accuracy nur für besten Baum berechnen (spart Zeit)
dt_best_depth = DecisionTreeClassifier(
    random_state=42,
    max_depth=best_depth,
    min_samples_split=10,
    min_samples_leaf=5
)
dt_best_depth.fit(X_train, y_train)

test_acc_best_depth = accuracy_score(y_test, dt_best_depth.predict(X_test))
print(f"Test-Accuracy (best depth): {test_acc_best_depth:.4f}")

## 3. Post-Pruning: Cost Complexity Pruning

Post-Pruning durch Cost Complexity: Der vollständige Baum wird zunächst trainiert, dann systematisch beschnitten (alpha-Variation) basierend auf Validierungs-Accuracy.

Dies ist ein reaktiver Ansatz: Nach dem Training wird das Modell optimiert durch sukzessives Entfernen von komplexen Ästen:

In [ ]:
# Cost Complexity Pruning (beschleunigt: Tune-Subset + subsampled alphas + early stopping)
TUNE_FRACTION = 0.3
MAX_ALPHAS = 20
PATIENCE = 3
RANDOM_STATE = 42

# Kleinere Teilmenge des Trainings für schnelleres Fitting der Kandidatenbäume
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train, test_size=1 - TUNE_FRACTION, stratify=y_train, random_state=RANDOM_STATE
)

# Basisbaum mit denselben Regularisierungsparametern wie vorher, für den Pruning-Pfad
dt_full = DecisionTreeClassifier(
    random_state=RANDOM_STATE,
    min_samples_split=10,
    min_samples_leaf=5
)
dt_full.fit(X_tune, y_tune)

# Pfad der Kostenkomplexität
path = dt_full.cost_complexity_pruning_path(X_tune, y_tune)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

# Alphas: nur >0, unique, subsampling auf MAX_ALPHAS Kandidaten
alphas = ccp_alphas[ccp_alphas > 0]
alphas = np.unique(alphas)
if len(alphas) > MAX_ALPHAS:
    idx = np.linspace(0, len(alphas) - 1, MAX_ALPHAS).astype(int)
    alphas = alphas[idx]

best_dt_pruned = None
best_alpha = None
best_val = -np.inf
no_improve = 0

selected_alphas = []
train_scores_pruned = []
val_scores_pruned = []

# Kandidatenbäume auf Tune-Subset trainieren, auf Val auswerten
for alpha in alphas:
    dt_pruned = DecisionTreeClassifier(
        random_state=RANDOM_STATE, ccp_alpha=alpha,
        min_samples_split=10, min_samples_leaf=5
    )
    dt_pruned.fit(X_tune, y_tune)

    train_acc = accuracy_score(y_tune, dt_pruned.predict(X_tune))
    val_acc = accuracy_score(y_val, dt_pruned.predict(X_val))

    selected_alphas.append(alpha)
    train_scores_pruned.append(train_acc)
    val_scores_pruned.append(val_acc)

    if val_acc > best_val + 1e-6:
        best_val = val_acc
        best_alpha = alpha
        best_dt_pruned = dt_pruned
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        break

# Visualisierung (nur Training/Validation für Speed)
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(selected_alphas, train_scores_pruned, marker='o', label='Training (tune subset)', linewidth=2)
ax.plot(selected_alphas, val_scores_pruned, marker='s', label='Validation', linewidth=2)
ax.set_xlabel('Alpha (Complexity Parameter)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Alpha (tune subset, subsampled)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Bestes alpha final auf dem vollen Trainingsset neu trainieren
best_dt_pruned = DecisionTreeClassifier(
    random_state=RANDOM_STATE, ccp_alpha=best_alpha,
    min_samples_split=10, min_samples_leaf=5
)
best_dt_pruned.fit(X_train, y_train)

test_acc_best_pruned = accuracy_score(y_test, best_dt_pruned.predict(X_test))

print(f"\nBestes alpha: {best_alpha:.6f}")
print(f"Training-Accuracy (tune subset, best alpha): {max(train_scores_pruned):.4f}")
print(f"Validierungs-Accuracy (best alpha): {best_val:.4f}")
print(f"Test-Accuracy (best alpha, retrained full train): {test_acc_best_pruned:.4f}")
print(f"Anzahl der Knoten (final): {best_dt_pruned.tree_.node_count}")

## Vergleich: Ungeprunter vs. Pre-Pruning vs. Post-Pruning

Abschließender Vergleich der drei Strategien: Baseline (ungeprunter Baum), Pre-Pruning (beste Tiefe), Post-Pruning (Cost Complexity).

Vergleichen wir Feature Importance, Baumtiefe, Anzahl Knoten und Test-Accuracy:

In [ ]:
# Trainiere den Baum mit optimaler Tiefe neu für Vergleich
dt_optimal_depth = DecisionTreeClassifier(
    random_state=RANDOM_STATE,
    max_depth=best_depth,
    min_samples_split=10,
    min_samples_leaf=5
)
dt_optimal_depth.fit(X_train, y_train)

# Vergleich der drei Modelle
models = {
    'Original\n(max_depth=20)': dt_classifier,
    f'Optimal Depth\n(max_depth={best_depth})': dt_optimal_depth,
    'Pruned': best_dt_pruned
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for (name, model), ax in zip(models.items(), axes):
    feat_imp = model.feature_importances_
    top_n = 15
    top_indices = np.argsort(feat_imp)[-top_n:][::-1]
    top_importances = feat_imp[top_indices]
    
    ax.barh(range(len(top_importances)), top_importances, color='steelblue')
    ax.set_yticks(range(len(top_importances)))
    ax.set_yticklabels([f'Feat {i}' for i in top_indices], fontsize=8)
    ax.set_xlabel('Importance')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

# Detaillierter Vergleich
print("\n" + "="*70)
print("MODELLVERGLEICH")
print("="*70)

for name, model in models.items():
    y_test_pred_model = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred_model)
    
    print(f"\n{name}")
    print(f"  Test-Accuracy: {test_acc:.4f}")
    print(f"  Baumtiefe: {model.get_depth()}")
    print(f"  Anzahl Knoten: {model.tree_.node_count}")
    print(f"  Anzahl Features genutzt: {np.sum(model.feature_importances_ > 0)}")

## 4. Modell speichern

Das beste Decision-Tree-Modell wird für den finalen Vergleich in Notebook 06 gespeichert.


In [ ]:
# Bestes Modell (nach Pruning)
model_path = models_dir / "dt_best_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(best_dt_pruned, f)

# Zusätzlich Metadaten speichern
metadata = {
    "model_type": "Decision Tree (Cost Complexity Pruning)",
    "best_alpha": best_alpha,
    "test_accuracy": test_acc_best_pruned,
    "tree_depth": best_dt_pruned.get_depth(),
    "n_nodes": best_dt_pruned.tree_.node_count,
}

metadata_path = models_dir / "dt_metadata.pkl"
with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

print(f"Decision Tree Modell und Metadaten gespeichert in {models_dir}")
print(f"  - Test Accuracy: {test_acc_best_pruned:.4f}")
print(f"  - Baumtiefe: {best_dt_pruned.get_depth()}")
print(f"  - Knoten: {best_dt_pruned.tree_.node_count}")
